# New Section

In [3]:
import math
import ipywidgets as widgets
from IPython.display import display, HTML

# If widgets don't render in Colab, uncomment:
# from google.colab import output
# output.enable_custom_widget_manager()

# ---------- CSS (UPDATED: black grid lines inside the 9 boxes) ----------
display(HTML("""
<style>
/* Outer outline around the board */
.ttt-frame{
  border: 4px solid #00A9E0;
  border-radius: 16px;
  padding: 14px;
  display: inline-block;
  width: max-content;
  background: #0b1320;
  box-shadow: 0 0 0 3px rgba(0,169,224,.25), 0 10px 26px rgba(0,0,0,.45);
}

/* Grid lines: the gap shows the background color */
.ttt-grid{
  background: #0a0a0a;     /* BLACK grid lines */
  padding: 8px;            /* thickness of the outside grid line */
  border-radius: 14px;
  width: max-content;
}

/* Cell buttons (no borders; lines come from gap) */
.ttt-cell button{
  border: 0 !important;
  border-radius: 10px !important;
  font-weight: 800 !important;
  font-size: 26px !important;
  color: #111 !important;
}

.ttt-cell button:hover:enabled{
  outline: 3px solid #FFD100;
  outline-offset: 0px;
}

/* Status banner */
.ttt-status{
  font-family: ui-sans-serif, system-ui, -apple-system, Segoe UI, Roboto, Arial;
  font-weight: 800;
  font-size: 16px;
  padding: 10px 12px;
  border-radius: 12px;
  border: 2px solid #BBBDBF;
  background: #111827;
  color: #E5E7EB;
  display: inline-block;
  margin-bottom: 10px;
}
.turn-x { color: #ff4d4d; }
.turn-o { color: #4CAF50; }
</style>
"""))

# ---------- Game logic ----------
WIN_COMBOS = [
    [0, 1, 2], [3, 4, 5], [6, 7, 8],
    [0, 3, 6], [1, 4, 7], [2, 5, 8],
    [0, 4, 8], [2, 4, 6]
]

def check_winner(board):
    for combo in WIN_COMBOS:
        a, b, c = combo
        if board[a] == board[b] == board[c] != " ":
            return board[a], combo
    return None, None

def is_full(board):
    return " " not in board

def minimax(board, is_maximizing):
    winner, _ = check_winner(board)
    if winner == "O": return 1
    if winner == "X": return -1
    if is_full(board): return 0

    if is_maximizing:
        best = -math.inf
        for i in range(9):
            if board[i] == " ":
                board[i] = "O"
                best = max(best, minimax(board, False))
                board[i] = " "
        return best
    else:
        best = math.inf
        for i in range(9):
            if board[i] == " ":
                board[i] = "X"
                best = min(best, minimax(board, True))
                board[i] = " "
        return best

def ai_best_move(board):
    best_score = -math.inf
    move = None
    for i in range(9):
        if board[i] == " ":
            board[i] = "O"
            score = minimax(board, False)
            board[i] = " "
            if score > best_score:
                best_score = score
                move = i
    return move

# ---------- UI ----------
X_COLOR = "#ff4d4d"
O_COLOR = "#4CAF50"
WIN_COLOR = "#FFD54F"
EMPTY_COLOR = "#f3f3f3"

def status_html(msg, who=None):
    if who == "X":
        return f'Your turn: <span class="turn-x">X</span> — {msg}'
    if who == "O":
        return f'AI turn: <span class="turn-o">O</span> — {msg}'
    return msg

board = [" "] * 9
game_over = False

status = widgets.HTML(value=f'<div class="ttt-status">{status_html("Click a square to start.", who="X")}</div>')

cells = []
for i in range(9):
    btn = widgets.Button(
        description=" ",
        layout=widgets.Layout(width="86px", height="86px"),
        style=widgets.ButtonStyle(button_color=EMPTY_COLOR),
        tooltip=f"Cell {i+1}"
    )
    btn.add_class("ttt-cell")
    btn._index = i
    cells.append(btn)

grid = widgets.GridBox(
    children=cells,
    layout=widgets.Layout(
        grid_template_columns="repeat(3, 86px)",
        grid_template_rows="repeat(3, 86px)",
        gap="6px"  # <-- grid line thickness BETWEEN cells (make 4px/6px/8px)
    )
)
grid.add_class("ttt-grid")

frame = widgets.Box([grid])
frame.add_class("ttt-frame")
frame.layout = widgets.Layout(display="inline-block")  # keep board snug

def set_cell(i, player):
    board[i] = player
    cells[i].description = player
    cells[i].disabled = True
    cells[i].style.button_color = X_COLOR if player == "X" else O_COLOR

def end_game(message, winning_combo=None):
    global game_over
    game_over = True
    status.value = f'<div class="ttt-status">{status_html(message)}</div>'
    for b in cells:
        b.disabled = True
    if winning_combo:
        for i in winning_combo:
            cells[i].style.button_color = WIN_COLOR

def on_click(btn):
    global game_over
    if game_over:
        return

    i = btn._index
    if board[i] != " ":
        return

    # Human
    set_cell(i, "X")
    winner, combo = check_winner(board)
    if winner:
        end_game("You win!", combo)
        return
    if is_full(board):
        end_game("Tie game.")
        return

    # AI
    status.value = f'<div class="ttt-status">{status_html("Thinking...", who="O")}</div>'
    ai_i = ai_best_move(board)
    set_cell(ai_i, "O")

    winner, combo = check_winner(board)
    if winner:
        end_game("AI wins!", combo)
        return
    if is_full(board):
        end_game("Tie game.")
        return

    status.value = f'<div class="ttt-status">{status_html("Pick your next square.", who="X")}</div>'

for b in cells:
    b.on_click(on_click)

def reset(_=None):
    global board, game_over
    board = [" "] * 9
    game_over = False
    for b in cells:
        b.description = " "
        b.disabled = False
        b.style.button_color = EMPTY_COLOR
    status.value = f'<div class="ttt-status">{status_html("Click a square to start.", who="X")}</div>'

reset_btn = widgets.Button(description="Restart", button_style="info")
reset_btn.on_click(reset)

display(widgets.VBox([status, frame, reset_btn]))
